# Geração de Dados Sintéticos — `raw.fornecedores_clientes`

**Objetivo:** Popular a tabela `raw.fornecedores_clientes` com **250 registros**.

**Regras de integridade:**
- Os **200 IDs já referenciados** na tabela fato `raw.transacoes_financeiras` devem estar presentes.
- Formatos de ID existentes (preservados do notebook de transações):
  - Numérico: `1000000`, `1000004`, … (77 IDs)
  - `FORN-XXXXX`: `FORN-00002`, … (67 IDs)
  - `ACC-XXXXX`: `ACC-00007`, … (56 IDs)
- Mais 50 novos registros são adicionados (para atingir 250 total).

**Reprodutibilidade:** `seed = 42`

In [ ]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÕES
# ============================================================
import pandas as pd
import numpy as np
import hashlib
import uuid
import os
import glob
import random
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

QTD_TOTAL     = 250
SOURCE_SYSTEM = 'ERP_CORPORATIVO'
SOURCE_ENTITY = 'parceiros'
INGESTION_ID  = str(uuid.uuid4())
INGESTION_TS  = datetime(2026, 1, 10, 9, 0, 0).strftime('%Y-%m-%dT%H:%M:%S.000Z')

#print(f'ingestion_id : {INGESTION_ID}')
#print(f'ingestion_ts : {INGESTION_TS}')

ingestion_id : 9479b9c7-bd2a-42e6-b8b0-e0b98c046704
ingestion_ts : 2026-01-10T09:00:00.000Z


In [ ]:
# ============================================================
# 2. LEITURA DOS IDs JÁ EXISTENTES NA TABELA FATO
# ============================================================
workspace     = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
fato_dir      = os.path.join(workspace, 'data', 'raw', 'transacoes_financeiras')
fato_csvs     = sorted(glob.glob(os.path.join(fato_dir, '*.csv')))

dfs_fato = [pd.read_csv(f, usecols=['id_fornecedor_raw']) for f in fato_csvs]
df_fato  = pd.concat(dfs_fato, ignore_index=True)

ids_existentes = sorted(df_fato['id_fornecedor_raw'].astype(str).unique())

# Classificar por padrão
ids_num  = [i for i in ids_existentes if i.isdigit()]
ids_forn = [i for i in ids_existentes if i.startswith('FORN')]
ids_acc  = [i for i in ids_existentes if i.startswith('ACC')]

#print(f'IDs na tabela fato — Numéricos: {len(ids_num)} | FORN: {len(ids_forn)} | ACC: {len(ids_acc)}')
#print(f'Total IDs existentes: {len(ids_existentes)}')

IDs na tabela fato — Numéricos: 77 | FORN: 67 | ACC: 56
Total IDs existentes: 200


In [3]:
# ============================================================
# 3. DADOS DE REFERÊNCIA PARA GERAÇÃO SINTÉTICA
# ============================================================

PREFIXOS_EMPRESAS = [
    'Tech', 'Data', 'Global', 'Prime', 'Alpha', 'Beta', 'Nexo', 'Vox',
    'Sigma', 'Apex', 'Nova', 'Flex', 'Core', 'Link', 'Smart', 'Pro',
    'Mega', 'Ultra', 'Max', 'Top', 'Net', 'Info', 'Soft', 'Cloud',
    'Digital', 'Connect', 'Power', 'Fast', 'Sure', 'Safe',
]
SUFIXOS_EMPRESAS = [
    'Solutions', 'Systems', 'Consulting', 'Services', 'Technologies',
    'Brasil', 'Group', 'Partners', 'Logística', 'Comercial',
    'Assessoria', 'Ltda', 'S.A.', 'Indústria', 'Distribuidora',
    'Engenharia', 'Contábil', 'Financeira', 'Holding', 'Gestão',
]
CIDADES_ESTADOS = [
    ('São Paulo', 'SP'), ('Rio de Janeiro', 'RJ'), ('Belo Horizonte', 'MG'),
    ('Curitiba', 'PR'), ('Porto Alegre', 'RS'), ('Salvador', 'BA'),
    ('Fortaleza', 'CE'), ('Recife', 'PE'), ('Manaus', 'AM'), ('Goiânia', 'GO'),
    ('Brasília', 'DF'), ('Campinas', 'SP'), ('Santos', 'SP'), ('Florianópolis', 'SC'),
]
TIPOS_LOGRADOURO = ['Rua', 'Av.', 'Alameda', 'Travessa', 'Rodovia']

def gerar_nome_empresa(rng_seed):
    r = random.Random(rng_seed)
    return f"{r.choice(PREFIXOS_EMPRESAS)} {r.choice(SUFIXOS_EMPRESAS)}"

def gerar_cnpj(n):
    """Gera CNPJ fictício no formato 14 dígitos (sem validação)."""
    return str(n).zfill(14)

def gerar_email(nome, idx):
    slug = nome.lower().replace(' ', '.').replace('/', '').replace('á','a') \
                .replace('ã','a').replace('ç','c').replace('é','e') \
                .replace('ê','e').replace('ó','o').replace('ô','o') \
                .replace('ú','u').replace('í','i')[:30]
    return f'contato{idx}@{slug}.com.br'

def gerar_endereco(idx):
    r = random.Random(idx + 1000)
    tipo = r.choice(TIPOS_LOGRADOURO)
    num  = r.randint(1, 9999)
    cidade, estado = r.choice(CIDADES_ESTADOS)
    return f'{tipo} Corporativa {num}, {cidade} - {estado}'

def gerar_hash(row_dict):
    campos = ['id_fornecedor_raw', 'nome_fornecedor', 'tipo_fornecedor', 'cnpj_cpf']
    conteudo = '|'.join(str(row_dict.get(c, '')) for c in campos)
    return hashlib.sha256(conteudo.encode('utf-8')).hexdigest()

print('Funções auxiliares definidas.')

Funções auxiliares definidas.


In [4]:
# ============================================================
# 4. GERAÇÃO DOS 200 REGISTROS EXISTENTES (da tabela fato)
# ============================================================

TIPOS_FORN = ['FORNECEDOR', 'CLIENTE', 'AMBOS']
# Numérico → FORNECEDOR | FORN → FORNECEDOR ou AMBOS | ACC → CLIENTE ou AMBOS

def tipo_para_id(id_str, idx):
    r = random.Random(idx * 7)
    if id_str.isdigit():
        return r.choice(['FORNECEDOR', 'FORNECEDOR', 'AMBOS'])
    elif id_str.startswith('FORN'):
        return r.choice(['FORNECEDOR', 'FORNECEDOR', 'AMBOS'])
    elif id_str.startswith('ACC'):
        return r.choice(['CLIENTE', 'CLIENTE', 'AMBOS'])
    return 'FORNECEDOR'

registros_existentes = []
for idx, id_forn in enumerate(ids_existentes, start=1):
    nome   = gerar_nome_empresa(idx)
    tipo   = tipo_para_id(id_forn, idx)
    cnpj   = gerar_cnpj(idx * 100 + 1000)
    email  = gerar_email(nome, idx)
    endere = gerar_endereco(idx)

    row = {
        'id_fornecedor_raw': id_forn,
        'nome_fornecedor':   nome,
        'tipo_fornecedor':   tipo,
        'cnpj_cpf':          cnpj,
        'contato':           email,
        'endereco':          endere,
    }
    row['raw_row_hash'] = gerar_hash(row)
    registros_existentes.append(row)

print(f'Registros baseados na tabela fato gerados: {len(registros_existentes)}')

Registros baseados na tabela fato gerados: 200


In [ ]:
# ============================================================
# 5. GERAÇÃO DOS 50 NOVOS REGISTROS
# ============================================================
# Novos IDs — seguindo os padrões já existentes, usando faixas numéricas
# superiores para não conflitar:
#   - 17 novos FORN-XXXXX  (a partir de FORN-00201)
#   - 16 novos ACC-XXXXX   (a partir de ACC-00201)
#   - 17 novos numéricos   (a partir de 1000201)

novos_forn = [f'FORN-{i:05d}' for i in range(201, 218)]
novos_acc  = [f'ACC-{i:05d}'  for i in range(201, 217)]
novos_num  = [str(1000200 + i) for i in range(1, 18)]

novos_ids = novos_forn + novos_acc + novos_num
assert len(novos_ids) == 50, f'Esperado 50 novos, gerado {len(novos_ids)}'

registros_novos = []
for offset, id_forn in enumerate(novos_ids, start=len(ids_existentes) + 1):
    nome   = gerar_nome_empresa(offset + 9999)
    tipo   = tipo_para_id(id_forn, offset)
    cnpj   = gerar_cnpj(offset * 100 + 5000)
    email  = gerar_email(nome, offset)
    endere = gerar_endereco(offset)

    row = {
        'id_fornecedor_raw': id_forn,
        'nome_fornecedor':   nome,
        'tipo_fornecedor':   tipo,
        'cnpj_cpf':          cnpj,
        'contato':           email,
        'endereco':          endere,
    }
    row['raw_row_hash'] = gerar_hash(row)
    registros_novos.append(row)

# print(f'Novos registros gerados: {len(registros_novos)}')

Novos registros gerados: 50


In [ ]:
# ============================================================
# 6. CONSOLIDAÇÃO E METADADOS
# ============================================================

todos_registros = registros_existentes + registros_novos

for seq, row in enumerate(todos_registros, start=1):
    row['ingestion_id']  = INGESTION_ID
    row['ingestion_ts']  = INGESTION_TS
    row['source_system'] = SOURCE_SYSTEM
    row['source_entity'] = SOURCE_ENTITY
    row['row_seq']       = seq

COLUNAS = [
    'id_fornecedor_raw', 'nome_fornecedor', 'tipo_fornecedor', 'cnpj_cpf',
    'contato', 'endereco', 'ingestion_id', 'ingestion_ts',
    'source_system', 'source_entity', 'row_seq', 'raw_row_hash',
]
df_forn = pd.DataFrame(todos_registros, columns=COLUNAS)

# print(f'Shape final: {df_forn.shape}')
df_forn.head()

Shape final: (250, 12)


,id_fornecedor_raw,nome_fornecedor,tipo_fornecedor,cnpj_cpf,contato,endereco,ingestion_id,ingestion_ts,source_system,source_entity,row_seq,raw_row_hash
0,1000000,Alpha Holding,FORNECEDOR,00000000001100,contato1@alpha.holding.com.br,"Rua Corporativa 3245, Florianópolis - SC",9479b9c7-bd2a-42e6-b8b0-e0b98c046704,2026-01-10T09:00:00.000Z,ERP_CORPORATIVO,parceiros,1,9e5fec7f34b762dc62bfa5ce4b27db056408cb7506ab28...
1,1000004,Fast Systems,FORNECEDOR,00000000001200,contato2@fast.systems.com.br,"Rodovia Corporativa 9760, Fortaleza - CE",9479b9c7-bd2a-42e6-b8b0-e0b98c046704,2026-01-10T09:00:00.000Z,ERP_CORPORATIVO,parceiros,2,2a2d4407057074586e7cc1683bdb685ed9fd74150b3671...
2,1000014,Vox Holding,FORNECEDOR,00000000001300,contato3@vox.holding.com.br,"Travessa Corporativa 9156, Salvador - BA",9479b9c7-bd2a-42e6-b8b0-e0b98c046704,2026-01-10T09:00:00.000Z,ERP_CORPORATIVO,parceiros,3,ccff100c1d717a16906484df2beee77eec225185d0660c...
3,1000015,Vox Comercial,FORNECEDOR,00000000001400,contato4@vox.comercial.com.br,"Travessa Corporativa 1743, Campinas - SP",9479b9c7-bd2a-42e6-b8b0-e0b98c046704,2026-01-10T09:00:00.000Z,ERP_CORPORATIVO,parceiros,4,e7cd1b4c1facdbe691962cd85ef3304dc78e25efbcb95d...
4,1000016,Top Logística,AMBOS,00000000001500,contato5@top.logistica.com.br,"Travessa Corporativa 6540, Manaus - AM",9479b9c7-bd2a-42e6-b8b0-e0b98c046704,2026-01-10T09:00:00.000Z,ERP_CORPORATIVO,parceiros,5,71ac3ad8055189c0de6e04f0e3dc9d11f0bba65c29de41...


In [ ]:
# ============================================================
# 7. VALIDAÇÕES
# ============================================================

assert len(df_forn) == QTD_TOTAL, f'Esperado {QTD_TOTAL}, gerado {len(df_forn)}'

ids_gerados = set(df_forn['id_fornecedor_raw'].astype(str))
ausentes    = [i for i in ids_existentes if i not in ids_gerados]
assert len(ausentes) == 0, f'IDs ausentes: {ausentes[:5]}'

assert df_forn['id_fornecedor_raw'].nunique() == QTD_TOTAL, 'IDs duplicados!'
assert df_forn['cnpj_cpf'].nunique() == QTD_TOTAL, 'CNPJs duplicados!'

# print('Todos os 200 IDs da tabela fato estão presentes.')
# print('Sem IDs duplicados.')
# print()
# print('Distribuição por tipo_fornecedor:')
# print(df_forn['tipo_fornecedor'].value_counts().to_string())
# print()
# print('Amostra dos novos registros:')
print(df_forn.tail(5)[['id_fornecedor_raw', 'nome_fornecedor', 'tipo_fornecedor']].to_string(index=False))

✔ Todos os 200 IDs da tabela fato estão presentes.
✔ Sem IDs duplicados.

Distribuição por tipo_fornecedor:
tipo_fornecedor
FORNECEDOR    127
AMBOS          70
CLIENTE        53

Amostra dos novos registros:
id_fornecedor_raw   nome_fornecedor tipo_fornecedor
          1000213     Smart Holding      FORNECEDOR
          1000214 Digital Solutions           AMBOS
          1000215  Power Financeira      FORNECEDOR
          1000216         Info S.A.           AMBOS
          1000217    Nova Solutions      FORNECEDOR


In [ ]:
# ============================================================
# 8. EXPORTAÇÃO PARA CSV
# ============================================================

output_dir  = os.path.join(workspace, 'data', 'raw', 'fornecedores_clientes')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'fornecedores_clientes.csv')
df_forn.to_csv(output_path, index=False, encoding='utf-8')

# print(f'Arquivo exportado: {output_path}')
# print(f'Total de registros: {len(df_forn)}')

Arquivo exportado: c:\Users\Adam\Documents\Repositorio\TCC\SCAP\data\raw\fornecedores_clientes\fornecedores_clientes.csv
Total de registros: 250
